# 🔥 TruthLens — Fine-Tune RoBERTa on Google Colab (GPU)

**Steps:**
1. Upload your `news.csv` when prompted
2. Run all cells (takes ~5-10 min on T4 GPU)
3. Download the trained model folder at the end
4. Extract to `model/bert_model/` in your project

**Make sure Runtime → Change runtime type → GPU (T4)**

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers torch tqdm

In [ ]:
# Step 2: Upload your news.csv
from google.colab import files
uploaded = files.upload()  # Click 'Choose Files' and select data/news.csv

In [ ]:
# Step 3: Train RoBERTa
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import os, json
from datetime import datetime

# ── Config ─────────────────────────────────────────────
MODEL_NAME = "roberta-base"
MAX_LEN    = 256
BATCH_SIZE = 16       # GPU can handle more
EPOCHS     = 3
LR         = 2e-5
SAVE_PATH  = "bert_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Dataset Class ─────────────────────────────────────
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Load Data ─────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv("news.csv").dropna(subset=["content", "label"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Dataset: {len(df)} rows | FAKE: {(df.label==0).sum()} | REAL: {(df.label==1).sum()}")

X = df["content"].tolist()
y = df["label"].tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(NewsDataset(X_train, y_train, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(NewsDataset(X_test, y_test, tokenizer, MAX_LEN),  batch_size=BATCH_SIZE)

# ── Model ─────────────────────────────────────────────
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)

# ── Train ─────────────────────────────────────────────
best_acc = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss = correct = total = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["label"].to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
        preds = out.logits.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        loop.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total*100:.1f}%")

    # Evaluate
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Eval"):
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            all_p.extend(out.logits.argmax(dim=1).cpu().numpy())
            all_l.extend(batch["label"].numpy())
    acc = accuracy_score(all_l, all_p)
    f1  = f1_score(all_l, all_p, average="weighted")
    print(f"\nEpoch {epoch+1}: Acc={acc*100:.2f}% F1={f1*100:.2f}%")
    print(classification_report(all_l, all_p, target_names=["FAKE","REAL"]))
    if acc > best_acc:
        best_acc = acc
        os.makedirs(SAVE_PATH, exist_ok=True)
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(f"  ✅ Best model saved! Acc: {acc*100:.2f}%")

print(f"\n🏆 Done! Best accuracy: {best_acc*100:.2f}%")

In [ ]:
# Step 4: Quick test
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch

tok = RobertaTokenizer.from_pretrained("bert_model")
mdl = RobertaForSequenceClassification.from_pretrained("bert_model").to(device).eval()

tests = [
    ("FAKE", "SHOCKING: Scientists PROVE 5G towers cause cancer and government is HIDING it!"),
    ("REAL", "The Federal Reserve raised interest rates by 25 basis points."),
    ("FAKE", "Bill Gates admits microchips inside COVID vaccines EXPOSED!"),
    ("REAL", "NASA James Webb Telescope captured new galaxy images."),
    ("FAKE", "Drinking bleach cures COVID doctors BANNED from telling TRUTH!"),
]
for exp, txt in tests:
    enc = tok(txt, max_length=256, padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(mdl(enc["input_ids"].to(device), enc["attention_mask"].to(device)).logits, dim=1).cpu().numpy()[0]
    pred = "REAL" if p[1]>p[0] else "FAKE"
    ok = "✅" if pred==exp else "❌"
    print(f"{ok} Expected:{exp} Got:{pred} Conf:{max(p)*100:.1f}% | {txt[:60]}")

In [ ]:
# Step 5: Download trained model
import shutil
shutil.make_archive("bert_model", 'zip', "bert_model")
files.download("bert_model.zip")
print("\n📥 Download complete!")
print("Extract bert_model.zip into your project's model/bert_model/ folder")
print("Then run: python app.py")